In [1]:
import pandas as pd
import numpy as np

In [2]:
emails = pd.read_csv('../../data/raw/emails.csv', low_memory=False)
emails.shape

(123389, 27)

Standardizing Column Names

In [3]:
emails.columns = (emails.columns.str.lower())
emails.columns

Index(['co_ref', 'time_to_renewal', 'crm_accreditation_completed',
       'crm_timely_completion', 'crm_progress_towards_accreditation',
       'crm_delays_in_accreditation', 'crm_contractor_suggested_leave',
       'crm_contractor_engagement', 'crm_contractor_sentiment',
       'crm_contractor_sentiment_score', 'crm_dts_or_ssip_mentioned',
       'crm_customer_payment_intention', 'crm_competitors_mentioned',
       'crm_membership_level', 'crm_platform_issues_raised',
       'crm_agent_chased_contractor', 'crm_agent_chase_count',
       'crm_accreditation_issues', 'crm_membership_overdue',
       'crm_auto_renewal_status', 'crm_dissatisified_with_renewal_price',
       'crm_customer_complained', 'crm_refund_mentioned',
       'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
       'crm_financial_hardship_mentioned', 'year'],
      dtype='object')

In [4]:
emails['time_to_renewal'].value_counts()

time_to_renewal
prior_year     40022
14_out         32493
45_out         28008
pre_renewal    22866
Name: count, dtype: int64

Checking duplicates

In [5]:
emails.duplicated().sum()

np.int64(0)

In [6]:
emails['co_ref'].isnull().sum()

np.int64(0)

In [7]:
emails['time_to_renewal'].value_counts()

time_to_renewal
prior_year     40022
14_out         32493
45_out         28008
pre_renewal    22866
Name: count, dtype: int64

Dropping irrelevant columns<br>
year -> can be derived

In [8]:
emails['year'].unique()

array([2025, 2026, 2024])

In [9]:
emails = emails.drop('year', axis=1)
emails= emails.drop('time_to_renewal', axis=1)
emails.shape

(123389, 25)

Handling missing values 

In [10]:
emails= emails.apply(lambda col: col.fillna("Unknown"))

Standardized Categorical values

In [11]:
emails['crm_delays_in_accreditation'].unique()

array(['Yes', 'No', 'Not Discussed',
       "Not Discussed (However, I must answer Yes or No, so I'll choose No as there's no clear indication of delays)",
       'Not Discussed is not applicable here as there is no mention of accreditation delays, however, the answer to whether there are any delays based on the content provided is: No',
       'Unknown',
       'Not Discussed is not applicable here, so I will default to: No',
       'Not Discussed but potentially yes as the customer is trying to contact Ian and was told he may be in later or the next day',
       'Not Discussed (However, considering the context is about a policy and a phone number with no answer, it might imply a delay or issue, but strictly speaking, delays in accreditation are not directly mentioned.)',
       "Not Discussed, however, I can infer that there might be a delay since it's been 27+ days with no call, so I will answer: Yes",
       'Not Discussed is not an option here so I will default to No',
       '"No

In [12]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    "Not Discussed (However, I must answer Yes or No, so I'll choose No as there's no clear indication of delays)": "No",

    "Not Discussed is not applicable here as there is no mention of accreditation delays, however, the answer to whether there are any delays based on the content provided is: No": "No",

    "Not Discussed (However, considering the context is about a policy and a phone number with no answer, it might imply a delay or issue, but strictly speaking, delays in accreditation are not directly mentioned.)": "Not Discussed",

    "Not Discussed, however, I can infer that there might be a delay since it's been 27+ days with no call, so I will answer: Yes": "Yes",

    '"Not Discussed (However, since there is no information about accreditation delays, the most fitting answer from the options provided is ""No"")"': "No",

    "Not Discussed is not applicable here as there is no information about delays, so the answer should be: No": "No",

    '"Not Discussed (However, since the question requires a Yes/No answer, I\'ll provide ""No"" as there\'s no mention of delays)"': "No",

    "Not Discussed is not an option for this prompt, I will choose No as there is no mention of delays": "No",

    "Not Discussed is not an option here so I will default to No": "No",

    "Not Discussed but potentially yes as a reason for the AR call": "Yes",

    "Not Discussed is not an option here so the most suitable answer is No": "No"
}
emails["crm_delays_in_accreditation"] = emails["crm_delays_in_accreditation"].replace(mapping)

In [13]:
emails['crm_contractor_engagement'].unique()

array(['Yes', 'No', 'Not Discussed', 'Unknown', ' the price is too high',
       ' indicating dissatisfaction with the service or support provided."',
       ' indicating dissatisfaction with the cost."',
       ' which they claim to have requested last year."',
       '"" indicating dissatisfaction with the service."',
       '"" making it not feasible to renew."',
       '"" implying they do not wish to renew their membership."',
       ' and they have chosen to drop clients that require the certification."',
       ' indicating uncertainty about continuing with the service."',
       '"" which led to a discussion about a one-off saving and the value of the SafeContractor membership."',
       ' citing time constraints and difficulties in responding to multiple emails."',
       ' leading to frustration and a desire to cancel."',
       ' finding the cost and effort ridiculous."',
       ' just asked for invoice""."',
       '"" which led them to consider canceling their accreditatio

In [14]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    ' indicating dissatisfaction with the service or support provided."': "No",
    '"" indicating dissatisfaction with the service."': "No",
    '"" making it not feasible to renew."': "No",
    '"" implying they do not wish to renew their membership."': "No",
    ' indicating uncertainty about continuing with the service."': "No",
    ' and they have chosen to drop clients that require the certification."': "No",
    ' citing it\'s not the right time."': "No",
    ' implying they want to leave due to an automatic renewal issue."': "No",

    ' which they claim to have requested last year."': "Unknown",
    ' irrelevant requests': "Unknown"
}
emails["crm_contractor_engagement"] = (emails["crm_contractor_engagement"].map(mapping).fillna("Unknown"))

In [15]:
emails['crm_contractor_sentiment'].unique()

array(['Neutral', 'Not Discussed', 'Satisfied', 'Dissatisfied', 'Unknown',
       ' and they are considering cancellation."', 'Yes',
       'Initially Dissatisfied, later Satisfied',
       'Initially Dissatisfied, later Neutral',
       ' and believes the requests for qualifications and RAMS are ridiculous."',
       ' and protecting people."',
       ' and no work gained from the membership."', 'No'], dtype=object)

In [16]:
mapping = {
    "Satisfied": "Satisfied",
    "Neutral": "Neutral",
    "Dissatisfied": "Dissatisfied",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    "Initially Dissatisfied, later Neutral": "Neutral",

    "Yes": "Dissatisfied",
    "No": "Dissatisfied",
    " and no work gained from the membership.\"": "Dissatisfied"
}

emails["crm_contractor_sentiment"] = (
    emails["crm_contractor_sentiment"]
    .map(mapping)
    .fillna("Unknown")
)

In [17]:
emails['crm_contractor_sentiment_score'].unique()


array(['50', 'Not Discussed', '80', '0', '40', '30', '20', '100', '31',
       '90', '39', '35', '60', '84', '37', '45', '38', '16', '83', '91',
       '82', '99', '95', '46', '97', '33', '42', '93', '43', '48',
       'Unknown', '70', '27', '25', 'Yes', 'Dissatisfied', '36', '55',
       '41', '10', 'Neutral', 'Satisfied', '49', '94', '98', '86', '85',
       '25.5', '57.5', '62.5', '87', '96', '27.5'], dtype=object)

Handling datatypes

In [18]:
# convert to numeric
emails["crm_contractor_sentiment_score"] = pd.to_numeric(
    emails["crm_contractor_sentiment_score"], errors="coerce"
)

# convert to int
emails["crm_contractor_sentiment_score"] = (
    emails["crm_contractor_sentiment_score"]
    .fillna(-1)
    .astype(int)
)

# convert score category
def score_to_category(score):
    if score == -1:
        return "Unknown"
    elif score <= 40:
        return "Dissatisfied"
    elif score <= 60:
        return "Neutral"
    elif score <= 100:
        return "Satisfied"
    else:
        return "Unknown"

emails["sentiment_category"] = emails["crm_contractor_sentiment_score"].apply(score_to_category)
emails["crm_contractor_sentiment_score"] = emails["crm_contractor_sentiment_score"].apply(
    lambda x: "Unknown" if x == -1
    else "Dissatisfied" if x <= 40
    else "Neutral" if x <= 60
    else "Satisfied"
)

# preserve "Not Discussed" from original column
emails.loc[
    emails["crm_contractor_sentiment"] == "Not Discussed",
    "sentiment_category"
] = "Not Discussed"

In [19]:
emails['crm_dts_or_ssip_mentioned'].unique()

array(['No', 'Yes', 'Unknown', 'Dissatisfied', '20', '30', '0', '50',
       '80', 'Not Discussed'], dtype=object)

In [20]:
emails["crm_dts_or_ssip_mentioned"] = emails["crm_dts_or_ssip_mentioned"].apply(
    lambda x: "Unknown" if pd.isna(x)
    else "Yes" if str(x).strip().lower() == "yes"
    else "Not Discussed" if str(x).strip().lower() == "not discussed"
    else "Unknown" if str(x).strip().lower() == "unknown"
    else "No"
)

In [21]:
emails['crm_customer_payment_intention'].unique()

array(['No', 'Not Discussed', 'Yes', 'Unknown', '20', '0', '30'],
      dtype=object)

In [22]:
emails["crm_customer_payment_intention"] = emails["crm_customer_payment_intention"].replace({"0": "No"})

In [23]:
emails['crm_agent_chase_count'].unique()

array(['1', '0', '3', '2', '4', '7', '9', '6', '5', '11', 'More than 5',
       '14', '8', 'Multiple', 'Many', 'A few',
       'Multiple ( exact number not specified)', 'Not specified', '10',
       'Several', 'Multiple times ( exact number not specified)', '19',
       '05-Jun', 'Lots (no specific number mentioned)',
       'A number of times (no specific number mentioned)',
       'Quite a few times (no exact number provided)', 'Every month',
       'Not explicitly stated, but implied to be multiple times', '13',
       'Unknown', 'Not Discussed',
       'Not applicable (only one attempt to contact)',
       'Not applicable (only one email)', 'Multiple times (at least 7-8)',
       'Multiple times weekly', 'Multiple (exact number not specified)',
       '2 (at least, as the agent mentioned sending emails and making a call)',
       '17', '10+', 'Multiple times (exact number not specified)',
       'Multiple times (at least 7 emails and several phone calls)',
       'Multiple times (i

In [24]:
emails["crm_agent_chase_count"].value_counts()

crm_agent_chase_count
0                                                                               37199
1                                                                               31223
2                                                                               23726
Unknown                                                                         11155
3                                                                               10397
                                                                                ...  
many                                                                                1
Multiple times (at least 5 emails and 2 phone calls)                                1
Not applicable (agent chased multiple times, but exact number not specified)        1
Not applicable (multiple instances of chasing)                                      1
A few times (no exact number specified)                                             1
Name: count, Length: 113, dtype:

In [25]:
def map_chase_count(val):
    if pd.isna(val):
        return "Unknown"
    
    val = str(val).lower()

    # numeric
    try:
        num = int(val)
        if num <= 1:
            return "Low"
        elif num <= 4:
            return "Medium"
        else:
            return "High"
    except:
        pass

    # text mapping
    if "one" in val:
        return "Low"
    
    elif any(word in val for word in ["few", "several", "monthly", "every month"]):
        return "Medium"
    
    elif any(word in val for word in ["multiple", "many", "numerous", "more than", "8+", "quite a few"]):
        return "High"
    
    elif any(word in val for word in ["unknown", "not specified", "not discussed"]):
        return "Unknown"
    
    return "Unknown"


emails["crm_agent_chase_count"] = emails["crm_agent_chase_count"].apply(map_chase_count)

In [26]:
emails['crm_membership_overdue'].unique()

array(['Yes', 'Not Discussed', 'No',
       ' as a ""Deem to Satisfy"" for their CHAS accreditation',
       ' despite being up to date."',
       ' which they requested to be removed and the invoice reissued."',
       ' and the manner in which it was executed."',
       ' preventing them from renewing their membership."',
       ' which is not currently included."',
       ' possibly because it was too early."',
       ' and the invoice amount not being reduced despite not conducting a full audit."',
       ' and also requested clarification on the renewal health and safety assessment."',
       ' which they claimed was damaging to their business."',
       ' indeed too busy""."',
       ' citing that the questions did not apply to him as he works alone."',
       ' leading to frustration and multiple calls to chase the audit."',
       '"" and was unhappy with the level of customer service."',
       ' and also had problems providing site-specific RAMS and evidence of refresher trai

In [27]:
emails["crm_membership_overdue"].value_counts()

crm_membership_overdue
Not Discussed                                                                                                    54110
Yes                                                                                                              30137
No                                                                                                               27937
Unknown                                                                                                          11155
 but was advised that it had saved and QS part 2 still needs completing."                                            1
 HS Direct."                                                                                                         1
 especially after a fatality incident where the membership meant nothing to the Health and Safety executive."        1
 working in accounts                                                                                                 1
Almost expired           

In [28]:
#values not matching predefined categories were treated as missing and imputed as “Unknown” to handle non-informative text
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}
emails["crm_membership_overdue"] = (
    emails["crm_membership_overdue"]
    .map(mapping)
    .fillna("Unknown")
)

In [29]:
to_drop = [ ' preventing them from renewing their membership."',
                                                                          ' possibly because it was too early."',
                                 ' with all questions in part one greyed out and part two showing as complete."',
                                                                                                  ' HS Direct."',
 ' especially after a fatality incident where the membership meant nothing to the Health and Safety executive."',
                                                                          ' which was later clarified as a typo',
                                                                                '"" and requested 1-1 support."',
              ' and ""C02-02 Confined Spaces Pre-Entry Checklist.doc"" which was a checklist and not accepted."',
                                                             ' and the agent had to submit it on their behalf."',
                                                                     '"" and needed assistance from the agent."']
emails = emails[~emails['crm_membership_overdue'].isin(to_drop)]

In [30]:
emails['crm_auto_renewal_status'].unique()

array(['0', '2', '1', ' rendering their current accreditation invalid."',
       'Not Discussed', 'No', 'Yes', ' potentially up to 8 months."',
       ' and over-querying of supplied evidence."', 'Unknown',
       ' and not being familiar with it."',
       ' and also had to provide additional information for the accreditation process."',
       ' but was helped to correct this by the agent."',
       ' including evidence of face masks and Respiratory Protective Equipment (RPE)."',
       ' and employer\'s liability."'], dtype=object)

In [31]:
emails["crm_auto_renewal_status"].value_counts()

crm_auto_renewal_status
0                                                                                  105135
Unknown                                                                             11155
2                                                                                    4702
1                                                                                    2348
Not Discussed                                                                          20
No                                                                                     14
Yes                                                                                     7
 rendering their current accreditation invalid."                                        1
 potentially up to 8 months."                                                           1
 and over-querying of supplied evidence."                                               1
 and not being familiar with it."                                           

In [32]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "1": "Yes",
    "0": "No",
    "2": "Unknown",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}

emails["crm_auto_renewal_status"] = (
    emails["crm_auto_renewal_status"]
    .map(mapping)
    .fillna("Unknown")
)

In [33]:
to_drop = ['and also had to provide additional information for the accreditation process."']
emails = emails[~emails['crm_auto_renewal_status'].isin(to_drop)]

In [34]:
emails['crm_dissatisified_with_renewal_price'].unique()

array(['No', 'Not Discussed', 'Yes', '0', '2', 'Unknown',
       'No actions or steps were taken by the agent in the provided email content.'],
      dtype=object)

In [35]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "1": "Yes",
    "0": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}

emails["crm_dissatisified_with_renewal_price"] = (
    emails["crm_dissatisified_with_renewal_price"]
    .map(mapping))

In [36]:
emails['crm_customer_complained'].unique()

array(['No', 'Yes', 'Unknown', '[Yes/No]', 'Not Discussed',
       'Not applicable (no email content provided)',
       'Not applicable (there is no email content to analyze)',
       'Not applicable (there is no content to analyze)',
       'Not applicable (there is no call transcript or email content to analyze)',
       'Not applicable (no conversation provided)',
       'Not applicable (there is no call mentioned in the email content)',
       'Not applicable (there is no call transcript provided)',
       'Not Applicable',
       'Not Applicable (since there is no call content provided)',
       'Not applicable (no call transcript provided)'], dtype=object)

In [37]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",
    "Not applicable (no email content provided)": "Not Applicable",
    "Not applicable (there is no content to analyze)": "Not Applicable",
    "Not applicable (there is no call transcript or email content to analyze)": "Not Applicable",
    "Not applicable (no conversation provided)": "Not Applicable",
    "Not applicable (there is no call mentioned in the email content)": "Not Applicable",
    "Not Applicable": "Not Applicable",
    "Not applicable (there is no call transcript provided)": "Not Applicable"
}

emails["crm_customer_complained"] = emails["crm_customer_complained"].map(mapping)

In [38]:
emails['crm_refund_mentioned'].unique()

array(['Yes', 'No', 'Unknown',
       ' feeling it implied their evidence was not legitimate',
       ' which provides laundry services for top London hotels."',
       ' which would dramatically affect their business."',
       ' and the additional cost for expedited review."',
       ' despite their accreditation and insurances being up to date."',
       ' which the agent was not aware of."',
       ' and also mentioned being overcharged and not receiving their certificate on time."',
       ' but the reason for their frustration was not explicitly stated."',
       '"" with all questions on part one greyed out and part two already showing as complete."',
       ' and then hung up."',
       ' but it was unclear if it was directed at the agent or not."',
       ' indicating dissatisfaction."',
       ' considering it ""highway robbery""."',
       ' and saw no value in membership since Homebase was no longer trading."',
       ' which led to a client turning them down for work."', '

In [39]:
def clean_refund_mentions(val):
    val_str = str(val).strip()
    
    #Check for "Not applicable" variations first
    if "not applicable" in val_str.lower():
        return "Not Applicable"
    map_to_yes=['but it was unclear if it was directed at the agent or not."',
                'considering it ""highway robbery""."',
                'and saw no value in membership since Homebase was no longer trading."',
                'making the renewal process pointless for their organization."']
    if any(variation in val_str for variation in map_to_yes):
        return "Yes"
    return val_str
emails['crm_refund_mentioned'] = emails['crm_refund_mentioned'].apply(clean_refund_mentions)

In [40]:
to_drop = ['""with all questions on part one greyed out and part two already showing as complete."']
emails = emails[~emails['crm_refund_mentioned'].isin(to_drop)]


In [41]:
emails['crm_negative_customer_experience'].unique()

array(['Yes', 'Unknown',
       ' and also expressed concerns about the audit process', 'No',
       '[Yes/No/Not Discussed]', ' causing them to lose work."',
       'Not Discussed', 'Not applicable (no email content provided)',
       'Not applicable (there is no email content to analyze)'],
      dtype=object)

In [42]:
to_drop = ['Not applicable (no email content provided)']
emails = emails[~emails['crm_negative_customer_experience'].isin(to_drop)]

In [43]:
emails['crm_dissatisfaction_with_support'].unique()

array(['No', 'Yes', 'Not Discussed',
       ' indicating a negative experience due to personal or health issues."',
       '"" which may impact their eligibility for projects."',
       ' invalid address',
       ' which led to a last-minute and urgent request to resolve the payment."',
       ' indicating a negative experience."',
       ' indicating a negative experience with the process."',
       ' indicating a negative experience with the accreditation process."',
       '"" which hindered their ability to update documents."',
       ' indicating a sense of urgency and frustration."',
       ' causing confusion and concern."', ' Nick."', 'Unknown',
       'The Contractor had a negative experience with the assigned work activity not matching their business, leading to delays and discussions with the auditor and customer care.',
       '"The Contractor experienced a negative customer experience due to an error in their accreditation status, which was incorrectly listed as ""consulta

In [44]:
def clean_refund_mentions(val):
    val_str = str(val).strip()
    
    #Check for variations 
    variations=["negative experience","negative customer experience"] 
    if any(variation in val_str.lower() for variation in variations):
        return "Yes"
    
    return val_str
emails['crm_dissatisfaction_with_support'] = emails['crm_dissatisfaction_with_support'].apply(clean_refund_mentions)

In [45]:
to_drop = ['Nick."','"" which hindered their ability to update documents."']
emails = emails[~emails['crm_dissatisfaction_with_support'].isin(to_drop)]

In [46]:
emails['crm_financial_hardship_mentioned'].unique()

array(['Yes', 'Not Discussed', 'No', ' city is required"" message."',
       'Unknown', ' bye""."', '[Yes/No/Not Discussed]',
       '"The Contractor expressed frustration due to the issue with ""24 hour locksmiths"" and the inconvenience with the payment link not working."',
       ' safety', 'Yes ',
       'Not applicable (there is no email content to analyze)'],
      dtype=object)

In [47]:
to_drop = [' bye""."', 'Not applicable (no email content provided)']
emails = emails[~emails['crm_financial_hardship_mentioned'].isin(to_drop)]


In [48]:
emails['crm_membership_level'].value_counts()

crm_membership_level
In progress                                                       45217
Accredited                                                        35847
Not Discussed                                                     29889
Unknown                                                           11155
Members only                                                        854
Standard                                                            162
Not Accredited                                                      104
In Progress                                                          23
Silver                                                               17
Gold                                                                 13
Premier                                                              10
Standard membership level                                             9
Membership (Standard)                                                 8
Accredited (Gold level)                    

In [49]:
#mapping for "Standard" and "Accredited" groups
membership_mapping = {
    "Standard membership level": "Standard",
    "Standard Membership": "Standard",
    "Standard package": "Standard",
    "Accredited (Silver level)": "Accredited"
}
emails['crm_membership_level'] = emails['crm_membership_level'].replace(membership_mapping)
print(emails['crm_membership_level'].value_counts())

crm_membership_level
In progress                                                       45217
Accredited                                                        35850
Not Discussed                                                     29889
Unknown                                                           11155
Members only                                                        854
Standard                                                            173
Not Accredited                                                      104
In Progress                                                          23
Silver                                                               17
Gold                                                                 13
Premier                                                              10
Membership (Standard)                                                 8
Accredited (Gold level)                                               7
Standard membership                        

In [50]:
emails.isnull().sum()[emails.isnull().sum() > 0]

crm_dissatisified_with_renewal_price    3
crm_customer_complained                 4
dtype: int64

Saving cleaned dataset

In [51]:
import os
os.makedirs('../../data/cleaned', exist_ok=True)
emails.to_csv('../../data/cleaned/emails_cleaned.csv', index=False)
print('Saved! Shape:', emails.shape)

Saved! Shape: (123385, 26)
